# Ballistic-diamond sweep analysis

Loads `sweep-summary.h5` (produced by `ingest.py`) and walks through the
primary observables: coverage, depth distributions, depth-vs-angle,
channeling fraction, and Sn-vs-Pb comparison. Edit `SUMMARY_PATH` below
and run the cells top-to-bottom.

In [ ]:
import sys
from pathlib import Path

# Allow importing the analysis package from the notebook's directory.
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from ballistic_analysis import load_summary, viz

# ---- edit me ----
SUMMARY_PATH = Path('sweep-summary.h5')
# -----------------

summary = load_summary(SUMMARY_PATH)
print(f'Loaded {summary.size} rows from {SUMMARY_PATH}')
print(f'  ok      : {(summary["ok"] == 1).sum()}')
print(f'  species : {sorted(set(summary["species"]))}')
print(f'  E (keV) : {sorted(set(summary["E_keV"]))}')
print(f'  angles  : {sorted(set(summary["angle_deg"]))}')
print(f'  T (K)   : {sorted(set(summary["T_K"]))}')

## 1. Coverage check

Every (species, E, angle, T) cell should have the same number of ensemble
members.  Anything missing here means failed/killed jobs -- worth
investigating before drawing conclusions.

In [ ]:
try:
    import pandas as pd
    have_pandas = True
except ImportError:
    have_pandas = False

cells = viz.summarize_cells(summary)
if have_pandas:
    df_cells = pd.DataFrame(cells)
    display(df_cells.sort_values(['species', 'E_keV', 'angle_deg', 'T_K']))
else:
    for i in range(len(cells['species'])):
        print(f"{cells['species'][i]} E={cells['E_keV'][i]:.0f} A={cells['angle_deg'][i]:.2f} T={cells['T_K'][i]:.0f}  n={cells['n'][i]}  mean={cells['depth_mean'][i]:.1f}  med={cells['depth_median'][i]:.1f}  std={cells['depth_std'][i]:.1f}")

## 2. Depth distributions stratified by angle

The whole study hinges on these.  If channeling works, the small-angle
histograms should have a long deep tail that the large-angle ones lack.

In [ ]:
# Pick a representative energy/temperature; loop or hand-edit to scan others.
for sp in sorted(set(summary['species'])):
    for E in sorted(set(summary['E_keV'])):
        fig = viz.depth_histogram(summary, sp, by='angle_deg', E_keV=E, T_K=300.0)
        plt.show()

In [ ]:
# Same plot but log-y to see the deep tail clearly.
fig = viz.depth_histogram(summary, 'sn', by='angle_deg', E_keV=60.0, T_K=300.0, log=True)
plt.show()

## 3. Median depth vs angle (with IQR)

Boil each cell down to a central depth and inter-quartile range, then plot
vs angle, one line per temperature.

In [ ]:
for sp in sorted(set(summary['species'])):
    for E in sorted(set(summary['E_keV'])):
        try:
            fig = viz.depth_vs_parameter(summary, sp, x='angle_deg', group='T_K', E_keV=E)
            plt.show()
        except ValueError as e:
            print(f'skip {sp} E={E}: {e}')

## 4. Median depth vs temperature

Does annealing actually push ions deeper, or is the channeling effect set
entirely by the strike geometry?  Pin angle, vary T.

In [ ]:
for sp in sorted(set(summary['species'])):
    try:
        fig = viz.depth_vs_parameter(summary, sp, x='T_K', group='angle_deg', E_keV=60.0)
        plt.show()
    except ValueError as e:
        print(f'skip {sp}: {e}')

## 5. Channeling fraction vs angle

Threshold should be a few times the SRIM mean range; pick it from the
histograms above (typical choice: where the non-channeling bulk ends).

In [ ]:
# Eyeball the histograms in cell 2 and adjust this threshold.
DEEP_THRESHOLD_A = 200.0

for sp in sorted(set(summary['species'])):
    fig = viz.channeling_fraction(summary, sp, threshold_A=DEEP_THRESHOLD_A)
    plt.show()

## 6. Sn vs Pb at matched (E, angle, T)

Heavier ion = more nuclear stopping per collision but lower velocity at
fixed KE.  At low energies (where electronic stopping is small) Pb should
stop shallower than Sn; the channel geometry effect should be similar.
Compare with the experimental observation.

In [ ]:
fig, axes = plt.subplots(1, len(set(summary['E_keV'])), figsize=(14, 4), sharey=True)
if not hasattr(axes, '__len__'):
    axes = [axes]
for ax, E in zip(axes, sorted(set(summary['E_keV']))):
    for sp, colour in (('sn', 'C0'), ('pb', 'C3')):
        sel = summary[(summary['species'] == sp)
                      & (summary['E_keV'] == E)
                      & (summary['T_K'] == 300.0)
                      & (summary['ok'] == 1)]
        if sel.size == 0:
            continue
        angles = sorted(set(sel['angle_deg']))
        med = [np.nanmedian(sel['depth'][sel['angle_deg'] == a]) for a in angles]
        ax.plot(angles, med, 'o-', color=colour, label=sp.upper())
    ax.set_xlabel('tilt [deg]')
    ax.set_title(f'E = {E:g} keV')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('median depth [A]')
axes[-1].legend()
fig.suptitle('Sn vs Pb median implantation depth (T = 300 K)')
fig.tight_layout()
plt.show()

## 7. Sanity checks

Things that should always be true; if any of these fail, something's off
with the simulation, not the analysis.

In [ ]:
ok = summary[summary['ok'] == 1]

# (a) Almost all ions should be thermal at the end of the 1 ns anneal.
#     kT @ 900 K = ~0.078 eV, so anything >> 1 eV is suspicious.
n_hot = (ok['ke_eV'] > 1.0).sum()
print(f'Ions with final KE > 1 eV: {n_hot} / {ok.size}  ({100*n_hot/ok.size:.2f}%)')
if n_hot:
    hot = ok[ok['ke_eV'] > 1.0]
    print('  examples (highest KE):')
    for r in hot[np.argsort(-hot['ke_eV'])[:5]]:
        print(f'    {r["species"]} E={r["E_keV"]:.0f} A={r["angle_deg"]:.2f} T={r["T_K"]:.0f} ens={r["ensemble"]}  ke={r["ke_eV"]:.2f} eV  z={r["z"]:.1f}')

# (b) No ion should be in the vacuum above the slab.  z_top_of_slab ~ zhi - 30 ~ 2492 A.
n_sky = (ok['depth'] < -10.0).sum()
print(f'\nIons left above the surface (depth < -10 A): {n_sky}')

# (c) Final position dispersion at angle=0 (best channeling) should be wide.
for sp in sorted(set(ok['species'])):
    sel = ok[(ok['species'] == sp) & (ok['angle_deg'] == 0.0)]
    if sel.size == 0:
        continue
    print(f'\n{sp.upper()} angle=0  depth p10/p50/p90 = '
          f'{np.nanpercentile(sel["depth"], 10):.1f} / '
          f'{np.nanpercentile(sel["depth"], 50):.1f} / '
          f'{np.nanpercentile(sel["depth"], 90):.1f}  [A]')

## 8. Per-ion XY scatter (where in the channel did they go?)

Plot final (x, y) of each ion, coloured by depth.  Crystal alignment
should show a cluster pattern reflecting the channel geometry.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8), sharex=True, sharey=True)
for row, sp in enumerate(['sn', 'pb']):
    for col, A in enumerate(sorted(set(summary['angle_deg']))):
        ax = axes[row, col]
        sel = ok[(ok['species'] == sp) & (ok['angle_deg'] == A) & (ok['E_keV'] == 60.0)]
        if sel.size:
            sc = ax.scatter(sel['x'], sel['y'], c=sel['depth'], cmap='viridis', s=14)
            plt.colorbar(sc, ax=ax, label='depth [A]')
        ax.set_title(f'{sp.upper()}  angle={A:g}')
        ax.set_aspect('equal')
for ax in axes[-1]:
    ax.set_xlabel('x [A]')
for ax in axes[:, 0]:
    ax.set_ylabel('y [A]')
fig.suptitle('Final ion (x, y) at E = 60 keV, T = 300 K')
fig.tight_layout()
plt.show()

## 9. Compare against your experimental measurement

Drop your experimental (E, angle, T, mean_depth, depth_error) data into
the dict below and overlay it onto the simulation curve.  This is the
money plot.

In [ ]:
# ---- edit me: paste experimental results here ----
experiment = {
    # (species, E_keV, T_K): list of (angle_deg, mean_depth_A, err_A)
    # example placeholder:
    # ('sn', 35.0, 300.0): [(0.0, 180.0, 20.0), (0.5, 120.0, 15.0), (2.0, 80.0, 10.0)],
}
# --------------------------------------------------

if not experiment:
    print('No experimental data registered yet -- edit the cell above.')
else:
    for (sp, E, T), points in experiment.items():
        sel = ok[(ok['species'] == sp) & (ok['E_keV'] == E) & (ok['T_K'] == T)]
        if sel.size == 0:
            print(f'no sim data for {sp} E={E} T={T}')
            continue
        angles_sim = sorted(set(sel['angle_deg']))
        med_sim = [np.nanmedian(sel['depth'][sel['angle_deg'] == a]) for a in angles_sim]
        iqr_sim = [(np.nanpercentile(sel['depth'][sel['angle_deg'] == a], 25),
                    np.nanpercentile(sel['depth'][sel['angle_deg'] == a], 75))
                   for a in angles_sim]
        lo = np.array([m - l for m, (l, h) in zip(med_sim, iqr_sim)])
        hi = np.array([h - m for m, (l, h) in zip(med_sim, iqr_sim)])

        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.errorbar(angles_sim, med_sim, yerr=[lo, hi], fmt='o-', capsize=3,
                    label='sim (median, IQR)')
        a_e, d_e, s_e = zip(*points)
        ax.errorbar(a_e, d_e, yerr=s_e, fmt='s', color='C3', capsize=3,
                    label='experiment')
        ax.set_xlabel('tilt [deg]')
        ax.set_ylabel('depth [A]')
        ax.set_title(f'{sp.upper()}  E={E:g} keV, T={T:g} K')
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        plt.show()